# 02 - Content-Based Filtering
**Unit 2: Feature Extraction and Similarity Retrieval**

Using KNN (k-Nearest Neighbors) to find similar user health profiles and extract suitable workout/diet plans.


In [3]:
import pandas as pd
import numpy as np
import os
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Get proper path to data file
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
DATA_PATH = os.path.join(NOTEBOOK_DIR, 'data', 'merged_fitness_data.csv')

if not os.path.exists(DATA_PATH):
    print(f"ERROR: {DATA_PATH} not found. Run merge_datasets.py first.")
else:
    df = pd.read_csv(DATA_PATH)
    
    # Handle missing values
    numeric_features = ['Age', 'Height', 'Weight']
    categorical_features = ['Gender', 'Chronic_Disease', 'Activity_Level', 'Dietary_Preference', 'Fitness_Goal']
    
    for col in numeric_features:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].median() if df[col].notna().any() else 25.0)
    for col in categorical_features:
        df[col] = df[col].astype(str).fillna('None')
    
    # 1. Setup Preprocessing Pipeline
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ])
    
    X = preprocessor.fit_transform(df[numeric_features + categorical_features])
    
    # 2. Train Model
    knn = NearestNeighbors(n_neighbors=5, metric='cosine')
    knn.fit(X)
    
    # 3. Simulate Query
    query_user = pd.DataFrame([{
        'Age': 28, 'Height': 175, 'Weight': 82, 'Gender': 'male',
        'Chronic_Disease': 'None', 'Activity_Level': 'Moderate', 
        'Dietary_Preference': 'No Preference', 'Fitness_Goal': 'weight_loss'
    }])
    
    query_vec = preprocessor.transform(query_user)
    distances, indices = knn.kneighbors(query_vec)
    
    print("\n--- Similarity Match Content ---")
    print(df.iloc[indices[0]][['source', 'diet_recommendation', 'exercise_plan']].head())


--- Similarity Match Content ---
          source   diet_recommendation                 exercise_plan
20705  dataset_5  Exercise Plan Type 2   Workout with HR=70, Temp=35
22893  dataset_5  Exercise Plan Type 1  Workout with HR=100, Temp=41
21543  dataset_5  Exercise Plan Type 1   Workout with HR=85, Temp=41
21690  dataset_5  Exercise Plan Type 2   Workout with HR=86, Temp=35
21885  dataset_5  Exercise Plan Type 0   Workout with HR=88, Temp=33
